### TEL354 - Minería de Datos
# Taller 1 - Análisis de Movilidad y Accesibilidad Urbana en Santiago mediante Técnicas de Agrupamiento

**Nombre:**

## Preparación del Entorno
### Instalación de Dependencias

In [ ]:
# Instalar las dependencias necesarias
# pip install geopandas scikit-learn matplotlib seaborn umap-learn

### Cargar librerías de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from shapely.geometry import Point, Polygon

# Librerías para clustering y reducción de dimensionalidad
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import umap

import warnings
warnings.filterwarnings('ignore')

---
## Parte 1: Carga y Exploración de Datos

En esta primera parte del taller trabajarás con datos geoespaciales de las comunas de Santiago y con un dataset de movilidad y accesibilidad urbana. Realizarás la carga de archivos, la exploración inicial y visualizaciones básicas.

> **Contexto:** La calidad del sistema de transporte público es un factor determinante de la desigualdad urbana. En Santiago, el acceso al metro, la frecuencia de servicios y los tiempos de viaje varían considerablemente entre comunas. Este taller analiza esas diferencias usando técnicas de minería de datos.

### 1.1 Geodatos Regionales

**Objetivo:** Cargar un archivo shapefile con la división comunal de Chile y examinar su estructura.

**Instrucciones:**
- Utiliza GeoPandas (`gpd`) para cargar el archivo `division_comunal.shp`.
- Almacena los datos en un DataFrame llamado `df`.
- Muestra las primeras filas del DataFrame para explorar su estructura.

**Resultado esperado:** Un GeoDataFrame con información geoespacial de las comunas (geometrías, nombres de provincias, etc.).

In [ ]:
# Escriba aquí su código

### 1.2 Geodatos de Santiago

**Objetivo:** Filtrar los datos para obtener solo las comunas de Santiago y convertirlos al sistema de coordenadas WGS84.

**Instrucciones:**
- Filtra las filas donde el nombre de la provincia sea `"Santiago"`.
- Convierte las geometrías al sistema de coordenadas WGS84 (`EPSG:4326`).
- Almacena el resultado en una variable llamada `df_stgo`.
- Muestra las primeras filas del DataFrame filtrado.

**Resultado esperado:** Un GeoDataFrame con únicamente las comunas de Santiago en coordenadas WGS84.

In [ ]:
# Escriba aquí su código

### 1.3 Visualización del Mapa Base de Santiago

**Objetivo:** Crear una visualización simple del mapa de las comunas de Santiago.

**Instrucciones:**
- Crea una figura con `plt.figure()` de tamaño adecuado.
- Utiliza el método `plot()` de `df_stgo` para dibujar el mapa.
- Añade un título y elimina los ejes de la visualización.

**Resultado esperado:** Un mapa que muestre los contornos de las comunas de Santiago.

In [ ]:
# Escriba aquí su código

### 1.4 Datos de Movilidad y Accesibilidad Urbana

**Objetivo:** Cargar y explorar el dataset de movilidad urbana de Santiago.

**Instrucciones:**
- Utiliza pandas para cargar el archivo `movilidad.csv` y almacenarlo en un DataFrame llamado `df_mov`.
- Muestra las primeras filas del DataFrame.
- Imprime un resumen que incluya: número total de registros, número de columnas y la lista de nombres de columnas.
- Utiliza `.describe()` para mostrar estadísticas descriptivas de las variables numéricas.

El archivo contiene las siguientes columnas por sector:

| Columna | Descripción |
|---|---|
| `comuna` | Nombre de la comuna |
| `longitud` | Coordenada geográfica longitudinal |
| `latitud` | Coordenada geográfica latitudinal |
| `dist_metro` | Distancia promedio al metro más cercano (km) |
| `frec_bus` | Frecuencia promedio de buses por hora |
| `tiempo_viaje` | Tiempo promedio de viaje al trabajo (min) |
| `indice_acceso` | Índice de accesibilidad compuesto (0–100) |
| `tasa_transbordo` | Proporción de viajes que requieren al menos un transbordo |
| `prop_auto` | Proporción de viajes realizados en automóvil privado |
| `prop_transporte_pub` | Proporción de viajes en transporte público |
| `nse` | Nivel socioeconómico promedio del sector (1=más bajo, 7=más alto) |

**Resultado esperado:** Comprensión de la estructura y distribución de las variables de movilidad.

In [ ]:
# Escriba aquí su código

### 1.5 Análisis de la Distribución del Índice de Accesibilidad

**Objetivo:** Analizar cómo se distribuye el índice de accesibilidad entre los sectores de Santiago.

**Instrucciones:**
- Crea una figura con dos subplots lado a lado.
- En el primer subplot, genera un histograma de la columna `indice_acceso` con estimación de densidad (KDE).
- En el segundo subplot, genera un histograma de `dist_metro` con KDE.
- Añade títulos adecuados a cada subplot y etiqueta los ejes.
- Agrega una cuadrícula en ambos subplots.

**Resultado esperado:** Visualizaciones que revelen la distribución del acceso al transporte público en Santiago.

In [ ]:
# Escriba aquí su código

### 1.6 Visualización Geoespacial del Índice de Accesibilidad

**Objetivo:** Crear un mapa de Santiago que muestre espacialmente el índice de accesibilidad de cada sector.

**Instrucciones:**
- Crea un mapa base usando `df_stgo` con color de relleno y contorno adecuados.
- Sobre el mapa, representa cada sector con un punto en sus coordenadas de longitud y latitud.
- Codifica el `indice_acceso` mediante colores usando un mapa de colores adecuado (p.ej. `'RdYlGn'`).
- Añade una barra de colores con etiqueta descriptiva.
- Agrega un título y elimina los ejes.

**Resultado esperado:** Un mapa de Santiago donde el color de cada punto refleja su nivel de accesibilidad al transporte público, revelando patrones espaciales de desigualdad en movilidad.

In [ ]:
# Escriba aquí su código

---
## Parte 2: Preprocesamiento y Transformación de Datos

**Objetivo general:** Preparar los datos de movilidad para el análisis de clustering, aplicando técnicas de agregación, normalización y transformación de variables.

### 2.1 Preprocesamiento de Datos

**Objetivo:** Agregar los datos por comuna, estandarizar variables y preparar el GeoDataFrame para la fusión posterior.

**Instrucciones:**

**Agregación por comuna:**
- Agrupa `df_mov` por la columna `'comuna'`.
- Calcula el **promedio** de las columnas: `'longitud'`, `'latitud'`, `'dist_metro'`, `'frec_bus'`, `'tiempo_viaje'`, `'indice_acceso'`, `'tasa_transbordo'`, `'prop_auto'`, `'prop_transporte_pub'` y `'nse'`.
- Reinicia el índice y almacena en `comuna_grouped`.

**Estandarización:**
- Define una lista `features` con las columnas: `'dist_metro'`, `'frec_bus'`, `'tiempo_viaje'`, `'indice_acceso'`, `'tasa_transbordo'`, `'prop_auto'`, `'prop_transporte_pub'`, `'nse'`.
- Instancia un `StandardScaler` y aplica `fit_transform` a las columnas de `features` en `comuna_grouped`.
- Crea un DataFrame `comuna_scaled` con los datos escalados y la columna `'comuna'`.

**Normalización de nombres para fusión:**
- Convierte los nombres de comunas en `comuna_grouped` a mayúsculas.
- Crea una copia `df_stgo_norm` de `df_stgo` y convierte `'NOM_COM'` a mayúsculas.

**Verificación:**
- Muestra las primeras filas de `comuna_grouped` y `comuna_scaled`.

In [ ]:
# Escriba aquí su código

### 2.2 Análisis de Correlación de Variables

**Objetivo:** Explorar las relaciones entre las variables de movilidad antes de aplicar clustering.

**Instrucciones:**
- Calcula la matriz de correlación entre todas las variables de `features` en `comuna_grouped`.
- Visualiza la matriz de correlación como un mapa de calor (`sns.heatmap`), mostrando los valores numéricos con 2 decimales.
- Añade un título descriptivo.

**Resultado esperado:** Un mapa de calor que muestre correlaciones entre variables de movilidad, lo que te orientará sobre la redundancia de información y la idoneidad de las variables para clustering.

In [ ]:
# Escriba aquí su código

---
## Parte 3: Análisis de Clustering

En esta parte aplicarás técnicas de clustering para identificar patrones de movilidad en las comunas de Santiago. Determinarás el número óptimo de clusters, implementarás cuatro algoritmos diferentes y compararás sus resultados.

### 3.1 Determinación del Número Óptimo de Clusters

**Objetivo:** Desarrollar una función que permita determinar el número ideal de clusters usando múltiples métricas de evaluación.

**Instrucciones:**

Crea una función `evaluar_clusters(df_scaled, max_k=10)` que:

1. **Método del Codo (Elbow Method):**
   - Calcula la inercia para valores de `k` desde 2 hasta `max_k`.
   - Genera un gráfico de línea con marcadores. Título: `"Método del Codo"`, ejes: `"Número de clusters"` / `"Inercia"`.

2. **Coeficiente de Silueta (Silhouette Score):**
   - Calcula el puntaje de silueta para cada valor de `k`.
   - Genera el gráfico correspondiente. Título: `"Coeficiente de Silueta"`, ejes: `"Número de clusters"` / `"Puntaje de Silueta"`.

3. **Índice Calinski-Harabasz:**
   - Calcula el puntaje CH para cada valor de `k`.
   - Genera el gráfico correspondiente. Título: `"Índice Calinski-Harabasz"`, ejes: `"Número de clusters"` / `"Puntaje CH"`.

- Organiza los tres gráficos en una fila con `subplots`.
- Aplica la función a `comuna_scaled[features]`.

**Resultado esperado:** Tres gráficos que orienten la selección del número óptimo de clusters. Deberás **justificar** el valor de `k` elegido en el análisis posterior.

In [ ]:
# Escriba aquí su código

### 3.2 Aplicación de Algoritmos de Clustering

**Objetivo:** Aplicar y comparar cuatro algoritmos de clustering a los datos de movilidad de las comunas de Santiago.

**Instrucciones:**

**K-Means:**
- Define el número óptimo de clusters basándote en el análisis de la sección 3.1.
- Aplica K-Means y almacena los resultados en la columna `'kmeans_cluster'` tanto en `comuna_scaled` como en `comuna_grouped`.

**Clustering Jerárquico (Agglomerative Clustering):**
- Evalúa los cuatro métodos de enlace: `'ward'`, `'complete'`, `'average'`, `'single'`.
- Para cada método: instancia `AgglomerativeClustering`, calcula el puntaje de silueta y genera un **dendrograma** usando `scipy.cluster.hierarchy`.
- Genera un gráfico de barras comparando los puntajes de silueta de cada método.
- Selecciona el mejor método de enlace y aplica el clustering final.
- Almacena resultados en la columna `'agg_cluster'` en `comuna_scaled` y `comuna_grouped`.

**DBSCAN:**
- Experimenta con distintos valores de `eps` y `min_samples`.
- Justifica brevemente los parámetros elegidos (puedes usar comentarios en el código).
- Almacena los resultados en la columna `'dbscan_cluster'` en ambos DataFrames.
- Considera que DBSCAN puede generar puntos de ruido etiquetados como `-1`.

**GMM (Gaussian Mixture Models):**
- Instancia `GaussianMixture` con el número óptimo de componentes.
- Almacena los resultados en la columna `'gmm_cluster'` en ambos DataFrames.

**Comparación Visual:**
- Crea una figura con **cuatro subplots** (2x2).
- En cada subplot, genera un scatter plot con coordenadas `longitud` y `latitud`, coloreando por cluster.
- Añade barras de color, títulos descriptivos y etiquetas de ejes.

**Resultado esperado:** Comparación visual de cómo cada algoritmo agrupa las comunas según sus patrones de movilidad.

In [ ]:
# Escriba aquí su código

### 3.3 Fusión con Datos Geoespaciales y Visualización en Mapa

**Objetivo:** Visualizar los resultados de clustering en mapas geoespaciales de Santiago.

**Instrucciones:**

**Preparación y fusión:**
- Crea una copia de `comuna_grouped` llamada `comuna_geo`.
- Combina `df_stgo_norm` con `comuna_geo` usando un *left join* por las columnas `'NOM_COM'` y `'comuna'`.
- Almacena el resultado en `df_stgo_merged`.

**Visualización en mapas:**
- Crea una figura con **cuatro subplots** organizados en una matriz 2x2.
- Para cada algoritmo (K-Means, Jerárquico, DBSCAN, GMM):
  - Genera un mapa coroplético de las comunas coloreado según la asignación de clusters.
  - Añade una leyenda, título descriptivo y elimina los ejes.

**Resultado esperado:** Un panel de cuatro mapas que permita comparar los patrones espaciales de agrupamiento identificados por cada método.

In [ ]:
# Escriba aquí su código

---
## Parte 4: Análisis de Perfiles de Movilidad por Cluster

En esta parte analizarás las características de movilidad de cada cluster identificado, construyendo perfiles que describan el comportamiento de transporte de cada grupo de comunas.

### 4.1 Caracterización de los Clusters

**Objetivo:** Analizar y visualizar las características de movilidad de cada cluster para comprender los distintos perfiles de accesibilidad en Santiago.

**Instrucciones:**

**Análisis estadístico (usando resultados de K-Means):**
- Agrupa `comuna_grouped` por `'kmeans_cluster'`.
- Calcula media, desviación estándar, mínimo, máximo y conteo para las variables: `'indice_acceso'`, `'dist_metro'`, `'tiempo_viaje'` y `'prop_transporte_pub'`.
- Almacena en `cluster_stats` y muéstralo.

**Visualización de distribuciones:**
- Genera **boxplots** para cada una de las cuatro variables anteriores, separados por cluster de K-Means.
- Organiza los cuatro boxplots en una figura 2x2.
- Añade títulos, etiquetas de ejes y cuadrículas.

**Mapa coroplético de accesibilidad:**
- Usa `df_stgo_merged` para generar un mapa que visualice el `indice_acceso` promedio por comuna.
- Utiliza un mapa de colores apropiado, añade leyenda, título y elimina los ejes.

**Resultado esperado:** Perfiles claros de movilidad para cada cluster que permitan nombrar e interpretar cada grupo (p.ej. "comunas de alta accesibilidad", "comunas periféricas con baja conectividad", etc.).

In [ ]:
# Escriba aquí su código

---
## Parte 5: Reducción de Dimensionalidad y Visualización Avanzada

En esta parte aplicarás técnicas de reducción de dimensionalidad para visualizar de manera efectiva los patrones multidimensionales presentes en los datos de movilidad.

### 5.1 Análisis de Componentes Principales (PCA)

**Objetivo:** Reducir la dimensionalidad de los datos con PCA y visualizar los resultados en relación con los clusters identificados.

**Instrucciones:**

**Aplicación de PCA:**
- Crea un objeto `PCA` con 2 componentes.
- Aplica `fit_transform` a `comuna_scaled[features]`.
- Almacena el resultado en `comuna_pca`.

**Creación de DataFrame de resultados:**
- Genera `pca_df` que incluya: nombres de comunas, coordenadas `PC1` y `PC2`, asignaciones de los cuatro algoritmos y valores de `indice_acceso` originales.

**Visualización:**
- Crea una figura 2x2 con scatter plots usando `PC1` y `PC2`:
  - Subplot 1: Colorear por clusters K-Means (paleta `'viridis'`).
  - Subplot 2: Colorear por clusters jerárquicos (paleta `'plasma'`).
  - Subplot 3: Colorear por clusters DBSCAN (paleta `'cividis'`).
  - Subplot 4: Colorear por `indice_acceso` (paleta `'RdYlGn'`).
- Añade cuadrículas suaves y títulos.

**Análisis de varianza explicada:**
- Imprime la varianza explicada por PC1 y PC2 (con 4 decimales).
- Calcula e imprime la varianza acumulada.

**Heatmap de loadings:**
- Crea un DataFrame con los componentes principales (filas: PC1, PC2; columnas: variables originales).
- Genera un mapa de calor con valores dentro de cada celda y título descriptivo.

**Resultado esperado:** Visualizaciones que permitan entender la proyección de datos en 2D mediante PCA y qué variables de movilidad contribuyen más a cada componente.

In [ ]:
# Escriba aquí su código

### 5.2 Aplicación de UMAP para Visualización No Lineal

**Objetivo:** Aplicar UMAP (Uniform Manifold Approximation and Projection) para obtener una proyección no lineal y revelar patrones que podrían no ser evidentes con PCA.

**Instrucciones:**

**Aplicación de UMAP:**
- Crea un objeto `UMAP` con una semilla fija para reproducibilidad (`random_state`).
- Aplica `fit_transform` a `comuna_scaled[features]`.
- Almacena el resultado en `embedding`.

**Creación de DataFrame de resultados:**
- Genera `umap_df` con las mismas columnas que `pca_df`, pero usando las coordenadas `UMAP1` y `UMAP2`.

**Visualización:**
- Crea una figura 2x2 con scatter plots usando `UMAP1` y `UMAP2`.
- Usa las mismas asignaciones de color y paletas que en PCA.
- Añade cuadrículas y títulos descriptivos.

**Resultado esperado:** Visualizaciones que permitan comparar cómo UMAP versus PCA proyecta los datos. UMAP suele preservar mejor la estructura local, lo que puede revelar agrupaciones más nítidas en datos de movilidad geoespacial.

In [ ]:
# Escriba aquí su código

---
## Parte 6: Análisis Geoespacial Integrado — Brechas de Movilidad

En esta sección final integrarás todos los elementos de los análisis anteriores para crear visualizaciones avanzadas sobre las brechas de accesibilidad al transporte público en Santiago.

### 6.1 Mapa de Calor de Déficit de Accesibilidad (Hot Spots)

**Objetivo:** Desarrollar un mapa de calor que muestre visualmente las zonas con mayor déficit de accesibilidad, usando estimación de densidad kernel ponderada por el índice de accesibilidad *invertido*.

**Instrucciones:**
- Crea una figura de matplotlib con tamaño adecuado.
- Como capa base, dibuja `df_stgo_merged` con color claro y bordes visibles.
- Para la capa KDE:
  - Extrae las coordenadas (`longitud`, `latitud`) de `df_mov`.
  - Usa `(100 - indice_acceso)` como pesos (zonas con bajo acceso tendrán mayor peso).
  - Implementa la estimación de densidad kernel con `scipy.stats.gaussian_kde`.
  - Ordena los puntos por densidad para mejor visualización.
  - Crea un scatter plot coloreado según densidad (paleta `'YlOrRd'`).
- Añade una barra de colores con etiqueta `"Densidad de Déficit"`.
- Agrega un título apropiado y desactiva los ejes.

**Resultado esperado:** Un mapa de calor que destaque las zonas de Santiago con mayor concentración de sectores de baja accesibilidad al transporte público.

In [ ]:
# Escriba aquí su código

---
## Parte 7: Preguntas de Análisis

Responde cada pregunta de forma argumentada, apoyándote en los resultados obtenidos en las partes anteriores. Cuando sea posible, cita métricas cuantitativas (puntajes de silueta, varianza explicada, estadísticas por cluster, etc.).

### 7.1 Análisis Sociourbano y de Movilidad

1. **Segregación de movilidad:** ¿Es posible identificar patrones de segregación en el acceso al transporte público en Santiago? ¿Qué comunas presentan mayor déficit y qué comunas tienen mejor conectividad? ¿Existe un patrón espacial claro (p.ej. comunas periféricas vs. centrales)?

2. **Relación movilidad–NSE:** Basándote en los resultados del clustering y en el análisis de perfiles (Parte 4), ¿existe una correlación entre el nivel socioeconómico de una comuna y su índice de accesibilidad? ¿Esta relación es lineal o más compleja?

3. **Macrozonas de movilidad:** Si tuvieras que dividir Santiago en exactamente **3 macrozonas** según sus características de movilidad, ¿cuáles serían y qué características particulares tendría cada una? ¿Coinciden con las divisiones administrativas o con patrones geográficos conocidos?

4. **Implicancias de política pública:** Propón al menos **dos medidas de política pública** que podrían reducir las brechas de accesibilidad identificadas. Fundamenta cada propuesta con evidencia derivada de los análisis realizados.

*[Escriba aquí su análisis...]*

### 7.2 Comparación de Algoritmos de Clustering

1. **Diferencias fundamentales:** ¿Qué diferencias observas entre los resultados de K-Means, Clustering Jerárquico, DBSCAN y GMM en este conjunto de datos? ¿Cuál captura mejor la heterogeneidad interna de las comunas? Apoya tu respuesta con puntajes de silueta u otras métricas calculadas.

2. **Idoneidad para datos geoespaciales:** ¿Cuál de los cuatro algoritmos consideras más adecuado para análisis de movilidad georreferenciada y por qué? Analiza críticamente las ventajas y limitaciones de cada método en este contexto específico.

3. **Interpretación del dendrograma:** Basándote en los dendrogramas generados en la Parte 3.2, ¿qué comunas forman los sub-clusters más cohesivos? ¿Tiene sentido intuitivo esta agrupación dado lo que sabes de Santiago?

4. **Limitaciones del análisis:** Señala al menos **dos limitaciones metodológicas** del análisis realizado y propón cómo podrían abordarse si tuvieras acceso a datos adicionales.

*[Escriba aquí su análisis...]*

### 7.3 PCA vs. UMAP

1. ¿Qué porcentaje de la varianza total explican las dos primeras componentes de PCA? ¿Es suficiente para representar fielmente los datos de movilidad? ¿Qué variables contribuyen más a PC1 y PC2, y qué interpretación tienen en términos de movilidad urbana?

2. Compara las visualizaciones de PCA y UMAP: ¿revelan estructuras similares o distintas? ¿En cuál de las proyecciones los clusters son más visibles y separables? Justifica tu respuesta.

*[Escriba aquí su análisis...]*